In [1]:
from ultralytics import YOLO

import torch
import torchvision.transforms as transforms

from torchvision.models import resnet18
import torchvision.models as models
from PIL import Image

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from tqdm import tqdm
import time

In [2]:
YOLO_MODEL = YOLO("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\runs\\detect\\runs\\YOLOv8_baseline\\weights\\best.pt")

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = cnn.fc.in_features
cnn.fc = torch.nn.Sequential(
    torch.nn.Linear(num_features, 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(128, 2)
)

cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\best_chicken_cnn_augument2.pth"))

cnn.to(DEVICE)

cnn.eval()

C:\Users\klanz\AppData\Local\Temp\ipykernel_20148\102316409.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\M

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [5]:
def classify_crop(crop):

    image = Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB))

    tensor = transform(image)

    tensor = tensor.unsqueeze(0)

    tensor = tensor.to(DEVICE)

    with torch.no_grad():

        prediction = cnn(tensor)

        prediction = prediction.argmax(1).item()

    return prediction

In [6]:
def door_decision(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    cnn_classes = [p["cnn_prediction"] for p in predictions]

    if 1 in cnn_classes:
        return "CLOSE"

    if 0 in cnn_classes:
        return "OPEN"

    return "CLOSE"

In [7]:
def door_decision_yolo(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    classes = [p["class"] for p in predictions]

    if 1 in classes:
        return "CLOSE"

    if 0 in classes:
        return "OPEN"

    return "CLOSE"

In [8]:
CLASS_NAMES = {
    0: "chicken",
    1: "not_chicken"
}

In [9]:
def run_yolo(image_path, conf=0.5):

    result = YOLO_MODEL.predict(
        source=str(image_path),
        conf=conf,
        verbose=False
    )[0]

    predictions = []

    for box in result.boxes:

        cls = int(box.cls.item())

        confidence = float(box.conf.item())

        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]

        predictions.append({

            "class": cls,
            "class_name": CLASS_NAMES[cls],
            "confidence": confidence,
            "bbox": [int(x1), int(y1), int(x2), int(y2)]

        })

    return predictions

In [10]:
def run_pipeline(image_path, conf=0.5):

    detections = run_yolo(image_path, conf)

    image = cv2.imread(str(image_path))

    final_predictions = []

    for det in detections:

        x1, y1, x2, y2 = det["bbox"]

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        cnn_prediction = classify_crop(crop)

        final_predictions.append({

            "yolo_prediction": det["class"],

            "cnn_prediction": cnn_prediction,

            "confidence": det["confidence"],

            "bbox": det["bbox"]

        })

    return final_predictions

In [11]:
def load_ground_truth(label_path):

    if not Path(label_path).exists():
        return []

    gt=[]

    with open(label_path) as f:

        for line in f:

            line=line.strip()

            if line=="":

                continue

            cls=int(line.split()[0])

            gt.append(cls)

    return gt

In [12]:
def ground_truth_decision(gt_classes):

    if 1 in gt_classes:
        return "CLOSE"

    if 0 in gt_classes:
        return "OPEN"

    return "CLOSE"

In [13]:
test_images = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset\\test\\images").glob("*"))

results = []

In [14]:
for image_path in test_images:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")

    gt = load_ground_truth(label_path)

    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()

    yolo_predictions = run_yolo(image_path)

    yolo_time = (time.perf_counter()-start)*1000

    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()

    pipeline_predictions = run_pipeline(image_path)

    pipeline_time = (time.perf_counter()-start)*1000

    pipeline_decision = door_decision(pipeline_predictions)

    results.append({

        "image": image_path.name,

        "ground_truth": gt_decision,

        "yolo": yolo_decision,

        "pipeline": pipeline_decision,

        "yolo_time_ms": yolo_time,

        "pipeline_time_ms": pipeline_time,

        "objects_gt": gt,

        "objects_yolo": [x["class"] for x in yolo_predictions],

        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [15]:
df = pd.DataFrame(results)

df.head(20)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,538.6206,84.4677,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,16.5213,20.4586,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,18.0906,26.6762,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,16.6163,21.0780,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,19.9409,42.6086,"[0, 0, 0]","[0, 0, 0, 0]","[0, 0, 0, 1]"
5,1085.jpeg,OPEN,OPEN,OPEN,21.6704,26.0838,[0],[0],[0]
6,109.jpeg,OPEN,OPEN,OPEN,21.6390,24.4842,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,23.8867,27.1635,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,16.9217,20.8988,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,26.5629,33.5263,"[0, 0, 0]","[0, 0]","[0, 0]"


In [16]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

import numpy as np

In [17]:
decision_map = {

    "OPEN":1,

    "CLOSE":0

}

In [18]:
def evaluate_system(df, prediction_column, time_column):

    gt = df["ground_truth"].map(decision_map)

    pred = df[prediction_column].map(decision_map)

    accuracy = accuracy_score(gt, pred)

    precision = precision_score(
        gt,
        pred,
        zero_division=0
    )

    recall = recall_score(
        gt,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        gt,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(gt, pred)

    tn, fp, fn, tp = cm.ravel()

    avg_time = df[time_column].mean()

    print("="*50)

    print(prediction_column)

    print("="*50)

    print(f"Accuracy : {accuracy:.4f}")

    print(f"Precision: {precision:.4f}")

    print(f"Recall   : {recall:.4f}")

    print(f"F1-score : {f1:.4f}")

    print()

    print(f"TP : {tp}")

    print(f"FP : {fp}")

    print(f"TN : {tn}")

    print(f"FN : {fn}")

    print()

    print(f"Średni czas: {avg_time:.2f} ms")

    return {

        "Accuracy":accuracy,

        "Precision":precision,

        "Recall":recall,

        "F1":f1,

        "TP":tp,

        "FP":fp,

        "TN":tn,

        "FN":fn,

        "Time":avg_time

    }

In [19]:
yolo_results = evaluate_system(

    df,

    "yolo",

    "yolo_time_ms"

)

yolo
Accuracy : 0.9886
Precision: 0.9958
Recall   : 0.9836
F1-score : 0.9897

TP : 479
FP : 2
TN : 389
FN : 8

Średni czas: 21.61 ms


In [20]:
pipeline_results = evaluate_system(

    df,

    "pipeline",

    "pipeline_time_ms"

)

pipeline
Accuracy : 0.9829
Precision: 0.9896
Recall   : 0.9795
F1-score : 0.9845

TP : 477
FP : 5
TN : 386
FN : 10

Średni czas: 29.62 ms


In [21]:
comparison = pd.DataFrame(

    [

        yolo_results,

        pipeline_results

    ],

    index=[

        "YOLO",

        "YOLO + CNN"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.988610,0.995842,0.983573,0.989669,479,2,389,8,21.60679
YOLO + CNN,0.982916,0.989627,0.979466,0.984520,477,5,386,10,29.61545


In [22]:
dangerous_yolo = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN")

]

In [23]:
dangerous_pipeline = df[

    (df["ground_truth"]=="CLOSE") &

    (df["pipeline"]=="OPEN")

]

In [24]:
print()

print("Krytyczne błędy")

print("----------------")

print("YOLO:",len(dangerous_yolo))

print("YOLO+CNN:",len(dangerous_pipeline))


Krytyczne błędy
----------------
YOLO: 2
YOLO+CNN: 5


In [25]:
improved = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN") &

    (df["pipeline"]=="CLOSE")

]

improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,19.3017,23.1166,[1],[0],[1]


In [26]:
empty_images = 0
false_detections = 0

for row in results:

    if len(row["objects_gt"]) == 0:

        empty_images += 1

        if len(row["objects_yolo"]) > 0:
            false_detections += 1

print("Puste obrazy:",empty_images)
print("Fałszywe detekcje:",false_detections)

if empty_images>0:

    print(
        "Odsetek:",
        false_detections/empty_images
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [27]:
fp_images=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["pipeline"]=="OPEN":

        fp_images.append(row["image"])

print("False Positive:",len(fp_images))

fp_images

False Positive: 5


['coyote__lila_WSU_Lynx_IMG_0965_jpg.rf.vqQg0CptK1hvCGBkRVla.jpg',
 'Image-88-8f30f6.jpg',
 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg',
 'raptor__raptor_043_jpg.rf.Zvj9EKGgySzsjUZnOEDi.jpg',
 'raptor__raptor_050_jpg.rf.duuf3tqrVk3ZzBLkisY3.jpg']

In [28]:
fn_images=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["pipeline"]=="CLOSE":

        fn_images.append(row["image"])

print("False Negative:",len(fn_images))

fn_images

False Negative: 10


['1054.jpeg',
 'neg_poultry__poultry_115_jpg.rf.n5StsXV8mhnHSm7Os8df.jpg',
 'neg_poultry__poultry_133_jpg.rf.Y9pO5YIarKaSPg8D9A7C.jpg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-FY33flUTFsva0xhQAxhZwgHaEK.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg',
 'OIP-tX7YgAqzRxQxMYkGe_FCtwHaFj.jpeg',
 'OIP-XHrr9AeiI8Egrt85dSx7mwHaFj.jpeg',
 'OIP-z_Y26rFvbazALq6sfXjgxwHaE8.jpeg']

In [29]:
fp_yolo=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["yolo"]=="OPEN":

        fp_yolo.append(row["image"])

len(fp_yolo)

fp_yolo

['Image-84-bd2f1b.jpg', 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [30]:
fn_yolo=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["yolo"]=="CLOSE":

        fn_yolo.append(row["image"])

len(fn_yolo)

fn_yolo

['neg_poultry__poultry_222_jpg.rf.g0AgDML3T5PV9uIHp0Xd.jpg',
 'neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-FY33flUTFsva0xhQAxhZwgHaEK.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg',
 'OIP-tX7YgAqzRxQxMYkGe_FCtwHaFj.jpeg',
 'OIP-z_Y26rFvbazALq6sfXjgxwHaE8.jpeg']

In [31]:

OUTPUT = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset_experiments")



In [32]:
image_path_dark = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\dark\\images").glob("*"))
image_path_night = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\night\\images").glob("*"))
image_path_occlusion = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\occlusion\\images").glob("*"))
image_path_motion_blur = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\motion_blur\\images").glob("*"))

results_dark = []
results_night = []
results_occlusion = []
results_motion_blur = []


In [33]:
for image_path in image_path_dark:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_dark.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [34]:
for image_path in image_path_night:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_night.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [35]:
for image_path in image_path_occlusion:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_occlusion.append({
        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [36]:
for image_path in image_path_motion_blur:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_motion_blur.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [37]:
df_dark = pd.DataFrame(results_dark)
df_night = pd.DataFrame(results_night)
df_occlusion = pd.DataFrame(results_occlusion)
df_motion_blur = pd.DataFrame(results_motion_blur)

df_dark.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,74.2806,83.0819,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,62.9232,22.9337,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,16.5247,20.0915,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,16.3901,18.8760,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,14.7595,29.9796,"[0, 0, 0]","[0, 0, 0]","[0, 1, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,17.9782,20.1493,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,15.1242,24.5419,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,14.4720,26.1908,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,13.5117,16.1747,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,13.1500,18.8685,"[0, 0, 0]","[0, 0]","[0, 0]"


In [38]:
df_night.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,17.4760,21.6064,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,12.4550,18.0604,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,11.8241,22.1321,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,13.6098,16.9911,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,13.6126,33.7835,"[0, 0, 0]","[0, 0, 0, 0]","[0, 1, 0, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,12.2897,17.1985,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,17.9604,17.1471,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,16.3677,24.7769,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,12.0856,18.9767,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,15.9699,29.1602,"[0, 0, 0]","[0, 0]","[0, 0]"


In [39]:
df_occlusion.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,19.4054,13.2423,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,12.8152,16.7758,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,13.7301,23.3743,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,13.9464,21.3755,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,14.0839,29.6910,"[0, 0, 0]","[0, 0, 0, 0]","[0, 0, 1, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,14.4253,20.8176,[0],"[0, 0]","[1, 0]"
6,109.jpeg,OPEN,OPEN,OPEN,18.1223,16.8678,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,15.3300,27.0709,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,17.0290,18.4411,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,13.6389,24.8490,"[0, 0, 0]","[0, 0]","[0, 0]"


In [40]:
df_motion_blur.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,16.3640,22.2317,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,11.7143,16.5195,[0],[0],[0]
2,1048.jpeg,OPEN,CLOSE,CLOSE,11.0847,12.1015,[0],[],[]
3,1053.jpeg,OPEN,OPEN,OPEN,12.2910,17.8108,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,CLOSE,CLOSE,11.3079,29.2457,"[0, 0, 0]",[1],[1]
5,1085.jpeg,OPEN,OPEN,CLOSE,13.9181,23.9479,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,16.2366,24.9674,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,12.4391,17.3799,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,19.2123,16.6279,[0],[0],[0]
9,1133.jpeg,OPEN,CLOSE,CLOSE,11.3439,11.8483,"[0, 0, 0]",[],[]


In [41]:
yolo_results1 = evaluate_system(
    df_dark,
    "yolo",
    "yolo_time_ms"
)
pipeline_results1 = evaluate_system(
    df_dark,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9829
Precision: 0.9876
Recall   : 0.9815
F1-score : 0.9846

TP : 478
FP : 6
TN : 385
FN : 9

Średni czas: 15.66 ms
pipeline
Accuracy : 0.9670
Precision: 0.9893
Recall   : 0.9507
F1-score : 0.9696

TP : 463
FP : 5
TN : 386
FN : 24

Średni czas: 24.05 ms


In [42]:
yolo_results2 = evaluate_system(
    df_night,
    "yolo",
    "yolo_time_ms"
)
pipeline_results2 = evaluate_system(
    df_night,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9658
Precision: 0.9831
Recall   : 0.9548
F1-score : 0.9688

TP : 465
FP : 8
TN : 383
FN : 22

Średni czas: 15.34 ms
pipeline
Accuracy : 0.9328
Precision: 0.9954
Recall   : 0.8830
F1-score : 0.9358

TP : 430
FP : 2
TN : 389
FN : 57

Średni czas: 22.86 ms


In [43]:
yolo_results3 = evaluate_system(
    df_occlusion,
    "yolo",
    "yolo_time_ms"
)
pipeline_results3 = evaluate_system(
    df_occlusion,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9590
Precision: 0.9913
Recall   : 0.9343
F1-score : 0.9619

TP : 455
FP : 4
TN : 387
FN : 32

Średni czas: 15.07 ms
pipeline
Accuracy : 0.9431
Precision: 0.9781
Recall   : 0.9179
F1-score : 0.9470

TP : 447
FP : 10
TN : 381
FN : 40

Średni czas: 22.92 ms


In [44]:
yolo_results4 = evaluate_system(
    df_motion_blur,
    "yolo",
    "yolo_time_ms"
)
pipeline_results4 = evaluate_system(
    df_motion_blur,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.8189
Precision: 0.9227
Recall   : 0.7351
F1-score : 0.8183

TP : 358
FP : 30
TN : 361
FN : 129

Średni czas: 14.64 ms
pipeline
Accuracy : 0.8417
Precision: 0.9780
Recall   : 0.7310
F1-score : 0.8367

TP : 356
FP : 8
TN : 383
FN : 131

Średni czas: 19.99 ms


In [45]:
comparison = pd.DataFrame(
    [
        yolo_results1,
        pipeline_results1
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.982916,0.987603,0.981520,0.984552,478,6,385,9,15.662968
YOLO + CNN,0.966970,0.989316,0.950719,0.969634,463,5,386,24,24.046970


In [46]:
comparison = pd.DataFrame(
    [
        yolo_results2,
        pipeline_results2

    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.965831,0.983087,0.954825,0.96875,465,8,383,22,15.335104
YOLO + CNN,0.932802,0.995370,0.882957,0.93580,430,2,389,57,22.862880


In [47]:
comparison = pd.DataFrame(
    [
        yolo_results3,
        pipeline_results3
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.958998,0.991285,0.934292,0.961945,455,4,387,32,15.065310
YOLO + CNN,0.943052,0.978118,0.917864,0.947034,447,10,381,40,22.918997


In [48]:
comparison = pd.DataFrame(
    [
        yolo_results4,
        pipeline_results4
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.818907,0.922680,0.735113,0.818286,358,30,361,129,14.643520
YOLO + CNN,0.841686,0.978022,0.731006,0.836663,356,8,383,131,19.991405


In [49]:
dangerous_yolo1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN")
]

In [50]:
dangerous_yolo2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN")
]

In [51]:
dangerous_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN")
]

In [52]:
dangerous_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN")
]

In [53]:
dangerous_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["pipeline"]=="OPEN")
]

In [54]:
dangerous_pipeline2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["pipeline"]=="OPEN")
]

In [55]:
dangerous_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["pipeline"]=="OPEN")
]

In [56]:
dangerous_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]

In [57]:
dangerous_both = df[
    (df["ground_truth"]=="CLOSE") & (df["yolo"]=="OPEN") & (df["pipeline"]=="OPEN")
]
dangerous_both1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") & (df_dark["yolo"]=="OPEN") & (df_dark["pipeline"]=="OPEN")
]
dangerous_both2 = df_night[
    (df_night["ground_truth"]=="CLOSE") & (df_night["yolo"]=="OPEN") & (df_night["pipeline"]=="OPEN")
]
dangerous_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") & (df_occlusion["yolo"]=="OPEN") & (df_occlusion["pipeline"]=="OPEN")
]
dangerous_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") & (df_motion_blur["yolo"]=="OPEN") & (df_motion_blur["pipeline"]=="OPEN")
]


In [58]:
comparison = pd.DataFrame(

    [   

        yolo_results,

        pipeline_results,

        yolo_results1,

        pipeline_results1,

        yolo_results2,

        pipeline_results2,

        yolo_results3,

        pipeline_results3,

        yolo_results4,

        pipeline_results4

    ],

    index=[

        "YOLO normal",
        "YOLO+CNN normal",
        "YOLO dark",
        "YOLO+CNN dark",
        "YOLO night",
        "YOLO+CNN night",
        "YOLO occlusion",
        "YOLO+CNN occlusion",
        "YOLO motion",
        "YOLO+CNN motion"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO normal,0.988610,0.995842,0.983573,0.989669,479,2,389,8,21.606790
YOLO+CNN normal,0.982916,0.989627,0.979466,0.984520,477,5,386,10,29.615450
YOLO dark,0.982916,0.987603,0.981520,0.984552,478,6,385,9,15.662968
YOLO+CNN dark,0.966970,0.989316,0.950719,0.969634,463,5,386,24,24.046970
YOLO night,0.965831,0.983087,0.954825,0.968750,465,8,383,22,15.335104
YOLO+CNN night,0.932802,0.995370,0.882957,0.935800,430,2,389,57,22.862880
YOLO occlusion,0.958998,0.991285,0.934292,0.961945,455,4,387,32,15.065310
YOLO+CNN occlusion,0.943052,0.978118,0.917864,0.947034,447,10,381,40,22.918997
YOLO motion,0.818907,0.922680,0.735113,0.818286,358,30,361,129,14.643520
YOLO+CNN motion,0.841686,0.978022,0.731006,0.836663,356,8,383,131,19.991405


In [59]:
print()
print("Krytyczne błędy, wpuszczenie drapieżnika")
print("----------------")
print("YOLO dark:",len(dangerous_yolo1),"     YOLO night:",len(dangerous_yolo2),"     YOLO occlusion:",len(dangerous_yolo3),"    YOLO motion:",len(dangerous_yolo4))
print("YOLO+CNN dark:",len(dangerous_pipeline1),"YOLO+CNN night:",len(dangerous_pipeline2),"YOLO+CNN occlusion:",len(dangerous_pipeline3),"YOLO+CNN motion:",len(dangerous_pipeline4))
print("BOTH dark:",len(dangerous_both1),"     BOTH night:",len(dangerous_both2),"     BOTH occlusion:",len(dangerous_both3),"    BOTH motion:",len(dangerous_both4))


Krytyczne błędy, wpuszczenie drapieżnika
----------------
YOLO dark: 6      YOLO night: 8      YOLO occlusion: 4     YOLO motion: 30
YOLO+CNN dark: 5 YOLO+CNN night: 2 YOLO+CNN occlusion: 10 YOLO+CNN motion: 8
BOTH dark: 3      BOTH night: 2      BOTH occlusion: 3     BOTH motion: 3


In [60]:
locking_chicken_yolo = df[
    (df["ground_truth"]=="OPEN") & ((df["yolo"]=="CLOSE"))
]
locking_chicken_yolo1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE"))
]
locking_chicken_yolo2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE"))
]
locking_chicken_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE"))
]
locking_chicken_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE"))
]
locking_chicken_pipeline = df[
    (df["ground_truth"]=="OPEN") & ((df["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["pipeline"]=="CLOSE"))
]

locking_chicken_both = df[
    (df["ground_truth"]=="OPEN") & ((df["yolo"]=="CLOSE") | (df["pipeline"]=="CLOSE"))
]
locking_chicken_both1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE") | (df_dark["pipeline"]=="CLOSE"))
]
locking_chicken_both2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE") | (df_night["pipeline"]=="CLOSE"))
]
locking_chicken_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE") | (df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE") | (df_motion_blur["pipeline"]=="CLOSE"))
]

In [61]:
print()
print("Niekrytyczne błędy, niewpuszczenie kur")
print("----------------")
print("YOLO dark:",len(locking_chicken_both1),"     YOLO night:",len(locking_chicken_both2),"     YOLO occlusion:",len(locking_chicken_both3),"    YOLO motion:",len(locking_chicken_both4))



Niekrytyczne błędy, niewpuszczenie kur
----------------
YOLO dark: 25      YOLO night: 57      YOLO occlusion: 41     YOLO motion: 138


In [62]:
data = [[len(dangerous_yolo), len(dangerous_yolo1), len(dangerous_yolo2), len(dangerous_yolo3), len(dangerous_yolo4)],
        [len(dangerous_pipeline), len(dangerous_pipeline1), len(dangerous_pipeline2), len(dangerous_pipeline3), len(dangerous_pipeline4)],
        [len(dangerous_both), len(dangerous_both1), len(dangerous_both2), len(dangerous_both3), len(dangerous_both4)]]
columns = ["normal", "dark", "night", "occlusion", "motion"]
index = ["yolo", "yolo+cnn", "both"]
table = pd.DataFrame(data, columns=columns, index=index)
tolatextable = table.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print("\\multicolumn{5}{c}{\\textbf{Krytyczne błędy, wpuszczenie przeciwnika}} ")
print(tolatextable)

\setlength{\tabcolsep}{6pt}
\multicolumn{5}{c}{\textbf{Krytyczne błędy, wpuszczenie przeciwnika}} 
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrr}
\toprule
 & normal & dark & night & occlusion & motion \\
\midrule
yolo & 2 & 6 & 8 & 4 & 30 \\
yolo+cnn & 5 & 5 & 2 & 10 & 8 \\
both & 1 & 3 & 2 & 3 & 3 \\
\bottomrule
\end{tabular}
\end{table}



In [63]:
data_locking = [[len(locking_chicken_yolo), len(locking_chicken_yolo1), len(locking_chicken_yolo2), len(locking_chicken_yolo3), len(locking_chicken_yolo4)],
    [len(locking_chicken_pipeline), len(locking_chicken_pipeline1), len(locking_chicken_pipeline2), len(locking_chicken_pipeline3), len(locking_chicken_pipeline4)],
    [len(locking_chicken_both), len(locking_chicken_both1), len(locking_chicken_both2), len(locking_chicken_both3), len(locking_chicken_both4)]]

columns_lock = ["normal", "dark", "night", "occlusion", "motion"]
index_lock = ["yolo", "yolo+cnn", "both"]

table2 = pd.DataFrame(data_locking, columns=columns, index=index)

tolatextable2 = table2.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print("\\multicolumn{5}{c}{\\textbf{Niekrytycznie błędy, niewpuszczenie kur}} ")
print(tolatextable2)

\setlength{\tabcolsep}{6pt}
\multicolumn{5}{c}{\textbf{Niekrytycznie błędy, niewpuszczenie kur}} 
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrr}
\toprule
 & normal & dark & night & occlusion & motion \\
\midrule
yolo & 8 & 9 & 22 & 32 & 129 \\
yolo+cnn & 10 & 24 & 57 & 40 & 131 \\
both & 12 & 25 & 57 & 41 & 138 \\
\bottomrule
\end{tabular}
\end{table}



In [64]:
latex_table = comparison.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print(latex_table)

\setlength{\tabcolsep}{6pt}
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrrrrrr}
\toprule
 & Accuracy & Precision & Recall & F1 & TP & FP & TN & FN & Time \\
\midrule
YOLO normal & 0.99 & 1.00 & 0.98 & 0.99 & 479 & 2 & 389 & 8 & 21.61 \\
YOLO+CNN normal & 0.98 & 0.99 & 0.98 & 0.98 & 477 & 5 & 386 & 10 & 29.62 \\
YOLO dark & 0.98 & 0.99 & 0.98 & 0.98 & 478 & 6 & 385 & 9 & 15.66 \\
YOLO+CNN dark & 0.97 & 0.99 & 0.95 & 0.97 & 463 & 5 & 386 & 24 & 24.05 \\
YOLO night & 0.97 & 0.98 & 0.95 & 0.97 & 465 & 8 & 383 & 22 & 15.34 \\
YOLO+CNN night & 0.93 & 1.00 & 0.88 & 0.94 & 430 & 2 & 389 & 57 & 22.86 \\
YOLO occlusion & 0.96 & 0.99 & 0.93 & 0.96 & 455 & 4 & 387 & 32 & 15.07 \\
YOLO+CNN occlusion & 0.94 & 0.98 & 0.92 & 0.95 & 447 & 10 & 381 & 40 & 22.92 \\
YOLO motion & 0.82 & 0.92 & 0.74 & 0.82 & 358 & 30 & 361 & 129 & 14.64 \\
YOLO+CNN motion & 0.84 & 0.98 & 0.73 & 0.84 & 356 & 8 & 383 & 131 & 19.99 \\
\bottomrule
\end{tabular}
\end{table}



In [65]:
improved = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN") &
    (df_dark["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,18.8979,17.6226,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,14.3612,18.1591,[1],"[0, 0]","[1, 1]"
833,raptor__gbif_raptor_00646_jpg.rf.IC8W8N97wC0S3...,CLOSE,OPEN,CLOSE,13.3988,18.2033,[1],[0],[1]


In [66]:
improved = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN") &
    (df_night["pipeline"]=="CLOSE")

]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
285,Image-51-eb7770.jpg,CLOSE,OPEN,CLOSE,10.9553,17.0498,[1],[0],[1]
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,15.2265,19.8202,[1],[0],[1]
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,11.8511,22.7356,[1],"[0, 0]","[1, 1]"
833,raptor__gbif_raptor_00646_jpg.rf.IC8W8N97wC0S3...,CLOSE,OPEN,CLOSE,15.6779,21.9188,[1],[0],[1]
836,raptor__gbif_raptor_00679_jpg.rf.IO9lWEg1PBD8V...,CLOSE,OPEN,CLOSE,21.0295,22.9014,[1],[0],[1]
840,raptor__gbif_raptor_00731_jpg.rf.DaBUw9KxmsnkZ...,CLOSE,OPEN,CLOSE,11.5075,19.0926,[1],[0],[1]


In [67]:
improved = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN") &
    (df_occlusion["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,17.7691,18.7612,[1],[0],[1]


In [68]:
improved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN") &
    (df_motion_blur["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
91,coyote__lila_AMMonitor_Camera_Traps_NEK-VB1_00...,CLOSE,OPEN,CLOSE,12.5009,16.8954,[1],[0],[1]
166,fox__gbif_fox_0058_jpg.rf.276TrYjTZ8l8fHW2cbDV...,CLOSE,OPEN,CLOSE,12.5286,18.4810,[1],[0],[1]
186,fox__gbif_fox_0454_jpg.rf.ET1jFfTjXFmpusKl9rJT...,CLOSE,OPEN,CLOSE,10.9879,23.6600,[1],[0],[1]
193,fox__gbif_fox_0540_jpg.rf.SviRhT8VBkdRDGFU1QO6...,CLOSE,OPEN,CLOSE,13.6339,16.8615,[1],[0],[1]
196,fox__gbif_fox_0615_jpg.rf.xGBhUyQrn1rpiysNE7WH...,CLOSE,OPEN,CLOSE,14.0184,20.0525,[1],[0],[1]
198,fox__gbif_fox_0722_jpg.rf.4EaY143B8mKPFC0058JJ...,CLOSE,OPEN,CLOSE,14.8439,19.3921,[1],[0],[1]
226,Image-111-ed55ed.jpg,CLOSE,OPEN,CLOSE,15.8302,22.1761,[1],[0],[1]
246,Image-28-e7f0ae.jpg,CLOSE,OPEN,CLOSE,12.9871,17.2081,"[1, 1]",[0],[1]
250,Image-31-1d307b.jpg,CLOSE,OPEN,CLOSE,15.7527,25.0483,[1],[0],[1]
252,Image-32-08b1fa.jpg,CLOSE,OPEN,CLOSE,13.7245,16.0653,[1],[0],[1]


In [69]:
deproved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]
deproved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
171,fox__gbif_fox_0117_jpg.rf.qu0oF3WRaQtW6UDSh1cN...,CLOSE,CLOSE,OPEN,14.4335,22.8922,[1],[1],[0]
245,Image-28-88c850.jpg,CLOSE,CLOSE,OPEN,22.3996,33.4138,[1],[1],[0]
327,Image-82-7d6856.jpg,CLOSE,CLOSE,OPEN,14.8400,19.1115,[1],[1],[0]
355,neg_empty__background_empty_043_jpg.rf.XPpXqOG...,CLOSE,CLOSE,OPEN,19.5717,24.0390,[],[1],[0]
874,raptor__raptor_027_jpg.rf.PzIAqRQVSQEonrCHFVSD...,CLOSE,CLOSE,OPEN,13.9007,19.3286,"[1, 1]",[1],[0]


In [70]:
empty_images0 = 0
false_detections0 = 0
for row in results_dark:

    if len(row["objects_gt"]) == 0:

        empty_images0 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections0 += 1

print("Puste obrazy:",empty_images0)
print("Fałszywe detekcje:",false_detections0)
if empty_images0>0:

    print(
        "Odsetek:",
        false_detections0/empty_images0
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [71]:
empty_images1 = 0
false_detections1 = 0

for row in results_night:

    if len(row["objects_gt"]) == 0:

        empty_images1 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections1 += 1

print("Puste obrazy:",empty_images1)
print("Fałszywe detekcje:",false_detections1)

if empty_images1>0:

    print(
        "Odsetek:",
        false_detections1/empty_images1
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [72]:
empty_images2 = 0
false_detections2 = 0

for row in results_occlusion:

    if len(row["objects_gt"]) == 0:

        empty_images2 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections2 += 1

print("Puste obrazy:",empty_images2)
print("Fałszywe detekcje:",false_detections2)

if empty_images2>0:

    print(
        "Odsetek:",
        false_detections2/empty_images2
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [73]:
empty_images3 = 0
false_detections3 = 0

for row in results_motion_blur:

    if len(row["objects_gt"]) == 0:

        empty_images3 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections3 += 1

print("Puste obrazy:",empty_images3)
print("Fałszywe detekcje:",false_detections3)

if empty_images3>0:

    print(
        "Odsetek:",
        false_detections3/empty_images3
    )

Puste obrazy: 37
Fałszywe detekcje: 4
Odsetek: 0.10810810810810811


In [74]:

#ZMIEŃ CONFIDENC POTEM NA 0,5 I PORÓWNAJ!!!!!!